In [49]:
# Cell 1: Environment Setup
!pip install pandas numpy scikit-learn matplotlib pyarrow -q

import pandas as pd
import numpy as np
import os, json, csv, time, random, logging
from datetime import date, datetime, timedelta

random.seed(42)
np.random.seed(42)
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
print("✓ Environment ready for: Dimensional Modeling and Star Schema")

✓ Environment ready for: Dimensional Modeling and Star Schema



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: C:\Users\9901359\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [50]:
# Cell 2: Generate Dataset
PRODUCTS = ['Laptop','Monitor','Keyboard','Mouse','Headset','Webcam','USB Hub','Desk Lamp','Speaker','Tablet']
REGIONS = ['North','South','East','West']
STATUSES = ['completed','pending','cancelled','refunded']

records = []
for i in range(10000):
    records.append({
        'order_id': f'ORD-{i:05d}',
        'product': random.choice(PRODUCTS),
        'category': PRODUCTS[i % len(PRODUCTS)].split()[0] if i < 50 else random.choice(['Electronics','Accessories','Home','Sports','Office']),
        'region': random.choice(REGIONS),
        'quantity': random.randint(1, 50),
        'unit_price': round(random.uniform(9.99, 999.99), 2),
        'order_date': str(date(2024,1,1) + timedelta(days=random.randint(0,364))),
        'status': random.choices(STATUSES, weights=[70,15,10,5])[0]
    })

df = pd.DataFrame(records)
df['revenue'] = (df['quantity'] * df['unit_price']).round(2)

print(f"✓ Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

✓ Dataset loaded: 10000 rows, 9 columns


,order_id,product,category,region,quantity,unit_price,order_date,status,revenue
0,ORD-00000,Monitor,Laptop,North,48,282.27,2024-04-24,completed,13548.96
1,ORD-00001,Monitor,Monitor,North,38,427.69,2024-01-16,completed,16252.22
2,ORD-00002,Mouse,Keyboard,North,36,206.84,2024-11-28,pending,7446.24
3,ORD-00003,USB Hub,Mouse,South,29,593.36,2024-01-04,pending,17207.44
4,ORD-00004,Keyboard,Headset,West,22,285.08,2024-04-20,refunded,6271.76


In [51]:
# Cell 3: Data Profiling
print('=' * 60)
print('DATA PROFILE: Dimensional Modeling and Star Schema')
print('=' * 60)
print(f'\nShape: {df.shape}')
print(f'\nColumn Types:\n{df.dtypes}')
print(f'\nNull Counts:\n{df.isnull().sum()}')
print(f'\nDuplicate Rows: {df.duplicated().sum()}')
print(f'\nNumeric Summary:\n{df.describe()}')
print(f'\nUnique Values:')
for col in df.columns:
    print(f'  {col}: {df[col].nunique()} unique')
print(f'\nMemory Usage: {df.memory_usage(deep=True).sum() / 1024:.1f} KB')

DATA PROFILE: Dimensional Modeling and Star Schema

Shape: (10000, 9)

Column Types:
order_id          str
product           str
category          str
region            str
quantity        int64
unit_price    float64
order_date        str
status            str
revenue       float64
dtype: object

Null Counts:
order_id      0
product       0
category      0
region        0
quantity      0
unit_price    0
order_date    0
status        0
revenue       0
dtype: int64

Duplicate Rows: 0

Numeric Summary:
           quantity    unit_price       revenue
count  10000.000000  10000.000000  10000.000000
mean      25.140600    503.807532  12629.256019
std       14.580209    287.654131  11033.664281
min        1.000000     10.050000     16.440000
25%       12.000000    254.082500   3529.290000
50%       25.000000    502.315000   9347.180000
75%       38.000000    751.495000  19358.000000
max       50.000000    999.970000  49685.500000

Unique Values:
  order_id: 10000 unique
  product: 10 unique
 

In [52]:
# Cell 4: Core Implementation — Dimensional Modeling and Star Schema

import pandas as pd
from datetime import date, datetime

# ============================================================
# Build a Star Schema from the flat orders DataFrame
# Star Schema = 1 Fact table + N Dimension tables
# ============================================================

# --- Step 1: Create Dimension Tables ---

# dim_product: unique products with surrogate keys
dim_product = df[['product', 'category']].drop_duplicates().reset_index(drop=True)
dim_product.index.name = 'product_key'
dim_product = dim_product.reset_index()
print('=== dim_product ===')
print(dim_product)

# dim_region: unique regions
dim_region = pd.DataFrame({'region': df['region'].unique()})
dim_region.index.name = 'region_key'
dim_region = dim_region.reset_index()
print('\n=== dim_region ===')
print(dim_region)

# dim_date: generate date dimension from order dates
df['order_date'] = pd.to_datetime(df['order_date'])
unique_dates = df['order_date'].drop_duplicates().sort_values().reset_index(drop=True)
dim_date = pd.DataFrame({
    'date_key': range(len(unique_dates)),
    'full_date': unique_dates,
    'year': unique_dates.dt.year,
    'month': unique_dates.dt.month,
    'quarter': unique_dates.dt.quarter,
    'day_of_week': unique_dates.dt.day_name(),
    'is_weekend': unique_dates.dt.dayofweek.isin([5, 6]).astype(int)
})
print(f'\n=== dim_date ({len(dim_date)} rows) ===')
print(dim_date.head(10))

# dim_status: order statuses
dim_status = pd.DataFrame({'status': df['status'].unique()})
dim_status.index.name = 'status_key'
dim_status = dim_status.reset_index()
print('\n=== dim_status ===')
print(dim_status)

# --- Step 2: Create Fact Table ---
# Map natural keys to surrogate keys
fact = df.copy()
fact = fact.merge(dim_product, on=['product', 'category'], how='left')
fact = fact.merge(dim_region, on='region', how='left')
fact = fact.merge(dim_date[['date_key', 'full_date']], left_on='order_date', right_on='full_date', how='left')
fact = fact.merge(dim_status, on='status', how='left')

fact_sales = fact[['order_id', 'product_key', 'region_key', 'date_key', 'status_key',
                    'quantity', 'unit_price', 'revenue']].copy()
print(f'\n=== fact_sales ({len(fact_sales)} rows) ===')
print(fact_sales.head(10))

# --- Step 3: SCD Type 2 — Slowly Changing Dimension ---
# Demonstrate how to track historical changes in a dimension

def scd2_upsert(dim_df, new_record, key_col, tracked_cols):
    """
    SCD Type 2 upsert: when tracked columns change, expire the old row
    and insert a new current row.
    """
    today = str(date.today())
    current = dim_df[(dim_df[key_col] == new_record[key_col]) & (dim_df['is_current'] == True)]

    if current.empty:
        # New record: insert with effective dates
        new_row = {**new_record, 'effective_date': today, 'expiry_date': '9999-12-31', 'is_current': True}
        dim_df = pd.concat([dim_df, pd.DataFrame([new_row])], ignore_index=True)
        print(f'  INSERT new: {new_record[key_col]}')
    else:
        old_row = current.iloc[0]
        changed = any(old_row[col] != new_record[col] for col in tracked_cols)
        if changed:
            # Expire old row
            idx = current.index[0]
            dim_df.at[idx, 'expiry_date'] = today
            dim_df.at[idx, 'is_current'] = False
            # Insert new current row
            new_row = {**new_record, 'effective_date': today, 'expiry_date': '9999-12-31', 'is_current': True}
            dim_df = pd.concat([dim_df, pd.DataFrame([new_row])], ignore_index=True)
            print(f'  UPDATE (SCD2): {new_record[key_col]} — region changed from {old_row["region"]} to {new_record["region"]}')
        else:
            print(f'  NO CHANGE: {new_record[key_col]}')
    return dim_df

# Demo SCD Type 2 with customer dimension
dim_customer = pd.DataFrame([
    {'customer_id': 1, 'name': 'Alice', 'region': 'North', 'effective_date': '2024-01-01', 'expiry_date': '9999-12-31', 'is_current': True},
    {'customer_id': 2, 'name': 'Bob', 'region': 'South', 'effective_date': '2024-01-01', 'expiry_date': '9999-12-31', 'is_current': True},
])

print('\n=== SCD Type 2 Demo ===')
print('Before:')
print(dim_customer)

# Alice moves from North to East
dim_customer = scd2_upsert(dim_customer, {'customer_id': 1, 'name': 'Alice', 'region': 'East'}, 'customer_id', ['region'])
# Bob stays the same
dim_customer = scd2_upsert(dim_customer, {'customer_id': 2, 'name': 'Bob', 'region': 'South'}, 'customer_id', ['region'])
# New customer
dim_customer = scd2_upsert(dim_customer, {'customer_id': 3, 'name': 'Charlie', 'region': 'West'}, 'customer_id', ['region'])

print('\nAfter SCD2 updates:')
print(dim_customer)

# --- Step 4: Star Schema Query Example ---
print('\n=== Star Schema Query: Revenue by Quarter and Region ===')
query_result = (fact_sales
    .merge(dim_date[['date_key', 'quarter']], on='date_key')
    .merge(dim_region, on='region_key')
    .groupby(['quarter', 'region'])
    .agg(total_revenue=('revenue', 'sum'), order_count=('order_id', 'count'))
    .round(2)
    .sort_values(['quarter', 'total_revenue'], ascending=[True, False]))
print(query_result)

print('\n-- Dimensional Modeling and Star Schema implementation complete')

=== dim_product ===
    product_key    product     category
0             0    Monitor       Laptop
1             1    Monitor      Monitor
2             2      Mouse     Keyboard
3             3    USB Hub        Mouse
4             4   Keyboard      Headset
..          ...        ...          ...
83           83    Headset         Home
84           84  Desk Lamp       Sports
85           85    Headset       Sports
86           86     Laptop  Electronics
87           87   Keyboard       Office

[88 rows x 3 columns]

=== dim_region ===
   region_key region
0           0  North
1           1  South
2           2   West
3           3   East

=== dim_date (365 rows) ===
   date_key  full_date  year  month  quarter day_of_week  is_weekend
0         0 2024-01-01  2024      1        1      Monday           0
1         1 2024-01-02  2024      1        1     Tuesday           0
2         2 2024-01-03  2024      1        1   Wednesday           0
3         3 2024-01-04  2024      1        1   

In [53]:
# Cell 5: Results Analysis
print('=' * 60)
print('RESULTS: Dimensional Modeling and Star Schema')
print('=' * 60)
print(f'Input rows: {len(df)}')
print(f'Processing complete')

# Display key metrics
print(f'\nRevenue by Region:')
print(df.groupby('region')['revenue'].sum().sort_values(ascending=False))

RESULTS: Dimensional Modeling and Star Schema
Input rows: 10000
Processing complete

Revenue by Region:
region
West     32704052.11
East     31929463.06
South    30902956.65
North    30756088.37
Name: revenue, dtype: float64


In [54]:
# Cell 6: Self-Check — Dimensional Modeling
# Run this cell to verify your work before clicking "Run Tests"
print('=' * 50)
print('SELF-CHECK — Dimensional Modeling')
print('=' * 50)

checks = {
    'DataFrame exists and is not empty': len(df) > 0,
    'Has at least 2 columns': len(df.columns) >= 2,
    'No fully-null columns': df.isnull().mean().max() < 0.5,
    'Has at least 10 rows': len(df) >= 10,
    'Data types look valid': df.dtypes is not None,
}

passed = sum(1 for v in checks.values() if v)
for name, ok in checks.items():
    print(f'  {"PASS" if ok else "FAIL"}: {name}')

print(f'\n{passed}/{len(checks)} self-checks passed')
if passed == len(checks):
    print('\nAll good! Click "Run Tests" to submit for official validation.')
else:
    print('\nSome checks failed. Fix your code, then click "Run Tests".')

SELF-CHECK — Dimensional Modeling
  PASS: DataFrame exists and is not empty
  PASS: Has at least 2 columns
  PASS: No fully-null columns
  PASS: Has at least 10 rows
  PASS: Data types look valid

5/5 self-checks passed

All good! Click "Run Tests" to submit for official validation.


In [55]:
import pandas as pd
from datetime import date

suppliers_oltp = pd.DataFrame([
    {'supplier_id': 1, 'name': 'SupplierA', 'region': 'North', 'discount_tier': 'Gold', 'contact_email': 'a@datamart.com'},
    {'supplier_id': 2, 'name': 'SupplierB', 'region': 'South', 'discount_tier': 'Silver', 'contact_email': 'b@datamart.com'}
])

dim_supplier = pd.DataFrame([
    {'supplier_id': 1, 'name': 'SupplierA', 'region': 'North', 'discount_tier': 'Gold', 'contact_email': 'a@datamart.com',
     'effective_date': '2024-01-01', 'expiry_date': '9999-12-31', 'is_current': True},
    {'supplier_id': 2, 'name': 'SupplierB', 'region': 'South', 'discount_tier': 'Silver', 'contact_email': 'b@datamart.com',
     'effective_date': '2024-01-01', 'expiry_date': '9999-12-31', 'is_current': True}
])

def scd2_upsert_supplier(dim_df, new_record, key_col, tracked_cols):
    today = str(date.today())
    current = dim_df[(dim_df[key_col] == new_record[key_col]) & (dim_df['is_current'] == True)]
    if current.empty:
        new_row = {**new_record, 'effective_date': today, 'expiry_date': '9999-12-31', 'is_current': True}
        dim_df = pd.concat([dim_df, pd.DataFrame([new_row])], ignore_index=True)
    else:
        old_row = current.iloc[0]
        changed = any(old_row[col] != new_record[col] for col in tracked_cols)
        if changed:
            idx = current.index[0]
            dim_df.at[idx, 'expiry_date'] = today
            dim_df.at[idx, 'is_current'] = False
            new_row = {**new_record, 'effective_date': today, 'expiry_date': '9999-12-31', 'is_current': True}
            dim_df = pd.concat([dim_df, pd.DataFrame([new_row])], ignore_index=True)
    return dim_df

dim_supplier = scd2_upsert_supplier(dim_supplier, {'supplier_id': 1, 'name': 'SupplierA', 'region': 'East', 'discount_tier': 'Gold', 'contact_email': 'a@datamart.com'}, 'supplier_id', ['region','discount_tier'])
dim_supplier = scd2_upsert_supplier(dim_supplier, {'supplier_id': 2, 'name': 'SupplierB', 'region': 'South', 'discount_tier': 'Platinum', 'contact_email': 'b@datamart.com'}, 'supplier_id', ['region','discount_tier'])
dim_supplier = scd2_upsert_supplier(dim_supplier, {'supplier_id': 3, 'name': 'SupplierC', 'region': 'West', 'discount_tier': 'Bronze', 'contact_email': 'c@datamart.com'}, 'supplier_id', ['region','discount_tier'])

print(dim_supplier)

checks = {
    'Has effective and expiry dates': all(col in dim_supplier.columns for col in ['effective_date','expiry_date']),
    'Has is_current flag': 'is_current' in dim_supplier.columns,
    'No duplicate current records': dim_supplier.groupby('supplier_id')['is_current'].sum().max() <= 1
}

for name, ok in checks.items():
    print(f'{"PASS" if ok else "FAIL"}: {name}')


   supplier_id       name region discount_tier   contact_email effective_date  \
0            1  SupplierA  North          Gold  a@datamart.com     2024-01-01   
1            2  SupplierB  South        Silver  b@datamart.com     2024-01-01   
2            1  SupplierA   East          Gold  a@datamart.com     2026-03-24   
3            2  SupplierB  South      Platinum  b@datamart.com     2026-03-24   
4            3  SupplierC   West        Bronze  c@datamart.com     2026-03-24   

  expiry_date  is_current  
0  2026-03-24       False  
1  2026-03-24       False  
2  9999-12-31        True  
3  9999-12-31        True  
4  9999-12-31        True  
PASS: Has effective and expiry dates
PASS: Has is_current flag
PASS: No duplicate current records
